# Módulo 4 · Clase 2 — Consultas básicas con SELECT

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Sentencia / cláusula | Para qué sirve |
|---|---------------------|----------------|
| 1 | `SELECT` básico | Elegir columnas |
| 2 | `WHERE` | Filtrar filas |
| 3 | `ORDER BY` | Ordenar resultados |
| 4 | `LIMIT` y `OFFSET` | Paginar resultados |
| 5 | `DISTINCT` | Eliminar duplicados |
| 6 | Funciones de agregación | `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` |
| 7 | Alias con `AS` | Nombres legibles en el resultado |
| 8 | Práctica | Explorar la base de datos de la tienda |

> 💡 Usaremos la base de datos `tienda.db` creada en la Clase 1. Ejecútala primero si no la tienes.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(Path.home() / "tienda.db")
conn.row_factory = sqlite3.Row   # acceso a columnas por nombre

def q(sql, *params):
    """Ejecuta una consulta y devuelve un DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)

# Verificar que las tablas existen
tablas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
print("Tablas disponibles:", tablas['name'].tolist())

---
## 1. `SELECT` básico

```sql
-- Todas las columnas
SELECT * FROM tabla;

-- Columnas específicas
SELECT col1, col2 FROM tabla;

-- Expresiones calculadas
SELECT nombre, precio * 1.21 AS precio_con_iva FROM productos;
```

La estructura básica de un `SELECT` completo sigue este **orden fijo**:

```sql
SELECT   columnas           -- ¿qué quiero ver?
FROM     tabla              -- ¿de dónde?
WHERE    condición          -- ¿qué filas?
ORDER BY columna [ASC|DESC] -- ¿en qué orden?
LIMIT    n OFFSET m;        -- ¿cuántas?
```

In [ ]:
# Todas las columnas
print("=== SELECT * ===")
display(q("SELECT * FROM clientes"))

# Columnas específicas
print("\n=== Columnas específicas ===")
display(q("SELECT nombre, email FROM clientes"))

# Expresión calculada: precio con IVA
print("\n=== Precio con IVA (21%) ===")
display(q("""
    SELECT nombre,
           precio                        AS precio_sin_iva,
           ROUND(precio * 1.21, 2)       AS precio_con_iva
    FROM productos
    LIMIT 5
"""))

---
## 2. `WHERE` — Filtrar filas

```sql
WHERE condición
```

**Operadores de comparación:**

| Operador | Significado | Ejemplo |
|----------|-------------|--------|
| `=` | Igual | `precio = 99.99` |
| `!=` / `<>` | Distinto | `estado != 'cancelado'` |
| `>`, `<`, `>=`, `<=` | Comparación numérica/textual | `precio > 100` |
| `BETWEEN a AND b` | Rango inclusivo | `precio BETWEEN 50 AND 200` |
| `IN (v1, v2, ...)` | En una lista | `ciudad IN ('Madrid', 'Sevilla')` |
| `NOT IN (...)` | Fuera de la lista | `estado NOT IN ('cancelado')` |
| `LIKE 'patrón'` | Coincidencia de texto | `nombre LIKE 'A%'` |
| `IS NULL` | Valor nulo | `descripcion IS NULL` |
| `IS NOT NULL` | No nulo | `email IS NOT NULL` |

**Patrones `LIKE`:**
- `%` → cualquier secuencia de caracteres (incluido ninguno)
- `_` → exactamente un carácter

**Operadores lógicos:** `AND`, `OR`, `NOT`

In [ ]:
# Comparación simple
print("=== Productos con precio < 50 ===")
display(q("SELECT nombre, precio FROM productos WHERE precio < 50 ORDER BY precio"))

# BETWEEN
print("\n=== Precio entre 50 y 200 ===")
display(q("SELECT nombre, precio FROM productos WHERE precio BETWEEN 50 AND 200"))

# IN
print("\n=== Pedidos pendientes o enviados ===")
display(q("SELECT id, cliente_id, estado FROM pedidos WHERE estado IN ('pendiente','enviado')"))

# LIKE
print("\n=== Productos cuyo nombre contiene 'Pro' ===")
display(q("SELECT nombre, precio FROM productos WHERE nombre LIKE '%Pro%'"))

# AND / OR
print("\n=== Electrónica con stock > 100 ===")
display(q("""
    SELECT nombre, precio, stock
    FROM productos
    WHERE categoria_id = 1
      AND stock > 100
"""))

---
## 3. `ORDER BY` — Ordenar resultados

```sql
ORDER BY columna1 [ASC|DESC], columna2 [ASC|DESC]
```

- `ASC` (por defecto): menor → mayor, A → Z
- `DESC`: mayor → menor, Z → A
- Puedes ordenar por varias columnas: primero por la primera, y en caso de empate por la segunda.

In [ ]:
# Más caros primero
print("=== Top 5 productos más caros ===")
display(q("""
    SELECT nombre, precio, stock
    FROM productos
    ORDER BY precio DESC
    LIMIT 5
"""))

# Ordenar por categoría y dentro de cada una por precio
print("\n=== Por categoría y precio ===")
display(q("""
    SELECT categoria_id, nombre, precio
    FROM productos
    ORDER BY categoria_id ASC, precio DESC
"""))

# Ordenar clientes por ciudad y nombre
print("\n=== Clientes ordenados ===")
display(q("SELECT nombre, ciudad FROM clientes ORDER BY ciudad, nombre"))

---
## 4. `LIMIT` y `OFFSET` — Paginar resultados

```sql
LIMIT n           -- devuelve solo los primeros n resultados
LIMIT n OFFSET m  -- salta los m primeros y devuelve los siguientes n
```

Muy útil para implementar paginación en aplicaciones:

```
Página 1: LIMIT 10 OFFSET 0
Página 2: LIMIT 10 OFFSET 10
Página 3: LIMIT 10 OFFSET 20
```

In [ ]:
PAGINA_SIZE = 3

for pagina in range(1, 5):
    offset = (pagina - 1) * PAGINA_SIZE
    resultado = q("""
        SELECT id, nombre, precio
        FROM productos
        ORDER BY precio DESC
        LIMIT ? OFFSET ?
    """, PAGINA_SIZE, offset)
    if resultado.empty:
        break
    print(f"Página {pagina}:")
    display(resultado)

---
## 5. `DISTINCT` — Eliminar duplicados

```sql
SELECT DISTINCT columna FROM tabla;
SELECT DISTINCT col1, col2 FROM tabla;  -- combinaciones únicas
```

In [ ]:
# Sin DISTINCT: puede haber ciudades repetidas
print("=== Ciudades (con duplicados) ===")
display(q("SELECT ciudad FROM clientes ORDER BY ciudad"))

# Con DISTINCT: cada valor una sola vez
print("\n=== Ciudades únicas ===")
display(q("SELECT DISTINCT ciudad FROM clientes ORDER BY ciudad"))

# DISTINCT con múltiples columnas: combinaciones únicas
print("\n=== Combinaciones únicas estado–cliente ===")
display(q("SELECT DISTINCT cliente_id, estado FROM pedidos ORDER BY cliente_id"))

---
## 6. Funciones de agregación

Las funciones de agregación calculan un único valor a partir de un conjunto de filas.

| Función | Devuelve | NULL |
|---------|----------|------|
| `COUNT(*)` | Número de filas | Cuenta todo |
| `COUNT(col)` | Filas donde col no es NULL | Ignora NULL |
| `SUM(col)` | Suma de los valores | Ignora NULL |
| `AVG(col)` | Media aritmética | Ignora NULL |
| `MIN(col)` | Valor mínimo | Ignora NULL |
| `MAX(col)` | Valor máximo | Ignora NULL |

```sql
-- Sin GROUP BY: calcula sobre TODA la tabla
SELECT COUNT(*), AVG(precio), MAX(precio) FROM productos;

-- Con GROUP BY (Clase 3): calcula por grupo
SELECT categoria_id, AVG(precio) FROM productos GROUP BY categoria_id;
```

In [ ]:
# Estadísticas generales de productos
print("=== Estadísticas de productos ===")
display(q("""
    SELECT
        COUNT(*)                AS total_productos,
        ROUND(AVG(precio), 2)   AS precio_medio,
        MIN(precio)             AS precio_minimo,
        MAX(precio)             AS precio_maximo,
        SUM(stock)              AS stock_total
    FROM productos
"""))

# COUNT diferencia *  vs columna
print("\n=== COUNT(*) vs COUNT(columna) ===")
display(q("""
    SELECT
        COUNT(*)           AS total_filas,
        COUNT(descripcion) AS con_descripcion
    FROM categorias
"""))

# Valor total del inventario
print("\n=== Valor total del inventario ===")
display(q("""
    SELECT
        COUNT(*)                          AS num_productos,
        ROUND(SUM(precio * stock), 2)     AS valor_inventario
    FROM productos
    WHERE stock > 0
"""))

# Precio medio de productos con precio > 100
print("\n=== Precio medio de productos premium (>100€) ===")
display(q("""
    SELECT ROUND(AVG(precio), 2) AS precio_medio_premium
    FROM productos
    WHERE precio > 100
"""))

---
## 7. Alias con `AS`

Los alias asignan un nombre temporal a una columna o expresión en el resultado.

```sql
SELECT nombre AS producto,
       precio AS coste,
       precio * stock AS valor_stock
FROM productos;

-- En SQLite, AS es opcional (pero es buena práctica incluirlo)
SELECT nombre producto FROM productos;
```

> Los alias **no pueden** usarse en `WHERE` (el motor evalúa `WHERE` antes de calcular las columnas). Sí pueden usarse en `ORDER BY`.

In [ ]:
# Alias en columnas calculadas
display(q("""
    SELECT
        nombre                          AS producto,
        precio                          AS precio_base,
        ROUND(precio * 1.21, 2)         AS precio_iva,
        stock                           AS unidades,
        ROUND(precio * stock, 2)        AS valor_en_almacen
    FROM productos
    ORDER BY valor_en_almacen DESC
    LIMIT 8
"""))

---
## 8. Práctica — Explorar la base de datos

---

### Ejercicio 1 — Filtros básicos

**Enunciado:** Escribe una consulta para cada caso:

1. Clientes de Madrid.
2. Productos con stock entre 50 y 150 unidades.
3. Pedidos del mes de enero de 2025 (pista: `LIKE '2025-01%'`).
4. Productos cuyo nombre empieza por vocal (A, E, I, O, U).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
# 1. Clientes de Madrid
display(q("SELECT nombre, email FROM clientes WHERE ciudad = 'Madrid'"))

# 2. Stock entre 50 y 150
display(q("SELECT nombre, stock FROM productos WHERE stock BETWEEN 50 AND 150 ORDER BY stock"))

# 3. Pedidos de enero 2025
display(q("SELECT id, cliente_id, fecha, estado FROM pedidos WHERE fecha LIKE '2025-01%'"))

# 4. Nombre empieza por vocal
display(q("""
    SELECT nombre FROM productos
    WHERE nombre LIKE 'A%'
       OR nombre LIKE 'E%'
       OR nombre LIKE 'I%'
       OR nombre LIKE 'O%'
       OR nombre LIKE 'U%'
    ORDER BY nombre
"""))
```
</details>

---
### Ejercicio 2 — Ordenar y paginar

**Enunciado:**

1. Muestra los tres productos con mayor valor de inventario (`precio × stock`).
2. Lista los clientes ordenados por ciudad (A→Z) y dentro de cada ciudad por nombre (A→Z).
3. Implementa una función `paginar(tabla, columna_orden, pagina, por_pagina=3)` que devuelva un DataFrame con la página correspondiente.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
# 1. Top 3 por valor de inventario
display(q("""
    SELECT nombre,
           precio,
           stock,
           ROUND(precio * stock, 2) AS valor_inventario
    FROM productos
    ORDER BY valor_inventario DESC
    LIMIT 3
"""))

# 2. Clientes ordenados
display(q("SELECT nombre, ciudad FROM clientes ORDER BY ciudad ASC, nombre ASC"))

# 3. Función de paginación
def paginar(tabla, columna_orden, pagina, por_pagina=3):
    offset = (pagina - 1) * por_pagina
    return q(f"""
        SELECT * FROM {tabla}
        ORDER BY {columna_orden}
        LIMIT {por_pagina} OFFSET {offset}
    """)

for p in range(1, 4):
    print(f"\nPágina {p}:")
    display(paginar("productos", "nombre", p))
```
</details>

---
### Ejercicio 3 — Agregación

**Enunciado:** Con una sola consulta, calcula:
- Número total de pedidos.
- Número de pedidos entregados.
- Número de pedidos pendientes.
- Fecha del pedido más antiguo.
- Fecha del pedido más reciente.

Pista: `SUM(estado = 'entregado')` funciona en SQLite (devuelve 1 o 0 por fila).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
display(q("""
    SELECT
        COUNT(*)                           AS total_pedidos,
        SUM(estado = 'entregado')          AS entregados,
        SUM(estado = 'pendiente')          AS pendientes,
        SUM(estado = 'enviado')            AS enviados,
        MIN(fecha)                         AS primer_pedido,
        MAX(fecha)                         AS ultimo_pedido
    FROM pedidos
"""))
```
</details>

---
### Ejercicio 4 — DISTINCT y combinaciones

**Enunciado:**

1. ¿Cuántas ciudades distintas hay en la tabla de clientes?
2. Lista las combinaciones únicas de (ciudad, primer carácter del nombre). Pista: `SUBSTR(nombre, 1, 1)` devuelve el primer carácter.
3. ¿Cuántos clientes han hecho pedidos? (pista: `COUNT(DISTINCT cliente_id)` sobre la tabla pedidos).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
# 1. Número de ciudades distintas
display(q("SELECT COUNT(DISTINCT ciudad) AS ciudades_distintas FROM clientes"))

# 2. Combinaciones únicas ciudad + inicial
display(q("""
    SELECT DISTINCT
        ciudad,
        SUBSTR(nombre, 1, 1) AS inicial
    FROM clientes
    ORDER BY ciudad, inicial
"""))

# 3. Clientes que han pedido
display(q("SELECT COUNT(DISTINCT cliente_id) AS clientes_con_pedido FROM pedidos"))
```
</details>

---
### Ejercicio 5 (reto) — Función `resumen_tienda`

**Enunciado:** Escribe la función `resumen_tienda(conn)` que imprima un dashboard completo de la tienda. Debe incluir:

- Total de clientes, productos y pedidos.
- Valor total del inventario (`precio × stock`).
- Precio medio, mínimo y máximo de los productos.
- Producto más caro y producto más barato.
- Ciudad con más clientes.
- Porcentaje de pedidos por estado.

Cada sección debe estar separada y etiquetada claramente.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import sqlite3, pandas as pd
from pathlib import Path

def resumen_tienda(conn):
    def q(sql):
        return pd.read_sql_query(sql, conn)

    print("=" * 50)
    print("    DASHBOARD — TIENDA ONLINE")
    print("=" * 50)

    # Totales
    totales = q("""
        SELECT
            (SELECT COUNT(*) FROM clientes)  AS clientes,
            (SELECT COUNT(*) FROM productos) AS productos,
            (SELECT COUNT(*) FROM pedidos)   AS pedidos
    """)
    print("\n--- Totales ---")
    display(totales)

    # Inventario
    inv = q("""
        SELECT
            ROUND(SUM(precio * stock), 2)  AS valor_inventario,
            ROUND(AVG(precio), 2)          AS precio_medio,
            MIN(precio)                    AS precio_min,
            MAX(precio)                    AS precio_max
        FROM productos
    """)
    print("\n--- Inventario ---")
    display(inv)

    # Producto más caro / más barato
    extremos = q("""
        SELECT nombre, precio, 'más caro' AS tipo
        FROM productos ORDER BY precio DESC LIMIT 1
        UNION ALL
        SELECT nombre, precio, 'más barato'
        FROM productos ORDER BY precio ASC LIMIT 1
    """)
    print("\n--- Extremos de precio ---")
    display(extremos)

    # Ciudad con más clientes
    ciudad = q("""
        SELECT ciudad, COUNT(*) AS clientes
        FROM clientes
        GROUP BY ciudad
        ORDER BY clientes DESC
        LIMIT 1
    """)
    print("\n--- Ciudad con más clientes ---")
    display(ciudad)

    # Pedidos por estado
    total = q("SELECT COUNT(*) AS n FROM pedidos").iloc[0]['n']
    estados = q("""
        SELECT estado,
               COUNT(*) AS pedidos
        FROM pedidos
        GROUP BY estado
        ORDER BY pedidos DESC
    """)
    estados['porcentaje'] = (estados['pedidos'] / total * 100).round(1).astype(str) + '%'
    print("\n--- Estados de pedidos ---")
    display(estados)

conn = sqlite3.connect(Path.home() / "tienda.db")
resumen_tienda(conn)
```
</details>

---
## Resumen de la clase

```sql
-- Estructura completa de un SELECT
SELECT  [DISTINCT]
        col1 [AS alias],
        ROUND(col2 * 1.21, 2) AS col_calculada,
        COUNT(*) AS total,
        AVG(precio) AS media
FROM    tabla
WHERE   precio BETWEEN 50 AND 500
  AND   nombre LIKE '%Pro%'
  AND   ciudad IN ('Madrid', 'Barcelona')
  AND   descripcion IS NOT NULL
ORDER BY precio DESC, nombre ASC
LIMIT 10 OFFSET 20;   -- página 3 de 10 en 10

-- Funciones de agregación
COUNT(*) · COUNT(col) · SUM(col) · AVG(col) · MIN(col) · MAX(col)
```

## ¿Qué viene en la próxima clase?

En la **Clase 3** combinamos tablas con `JOIN`, agrupamos resultados con `GROUP BY` y filtramos grupos con `HAVING`.

In [ ]:
conn.close()
print("Conexión cerrada")